<a href="https://colab.research.google.com/github/Rohan46os/50M-Model/blob/main/BoomLLm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import requests
import tiktoken

In [5]:
Url = "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"
data = requests.get(Url).text

In [6]:
len(data)

1115394

In [7]:
enc = tiktoken.get_encoding("o200k_base")
encoded_tokens = enc.encode(data)

In [8]:
len(encoded_tokens)

297606

In [9]:
batch_size = 4
sequence_length = 1024
batch_sequence = []

for i in range(batch_size):
  starting_index = i * sequence_length
  ending_index = sequence_length + starting_index
  batch_sequence.append(encoded_tokens[starting_index:ending_index])
  torched_sequence = torch.tensor(batch_sequence)

In [10]:
torched_sequence.size()

torch.Size([4, 1024])

In [11]:
vocab_size = 199997
d_embed = 64
theta = 1000

In [12]:
Embedding_table_ = nn.Embedding(vocab_size, d_embed)
Embedded_table = Embedding_table_(torched_sequence)

In [13]:
Embedded_table.size()

torch.Size([4, 1024, 64])

In [16]:
# Rope Embedding
x = Embedded_table #shape[4, 1024, 64]
# x = [] # word embedding
position_of_tokens = torch.arange(0, sequence_length).unsqueeze(1)#shape[1,1024]
frequencies = 1 / theta ** (2 * torch.arange(0, d_embed // 2) / d_embed)
cos_frequencies = torch.cos(frequencies*position_of_tokens).unsqueeze(0)
sin_frequencies = torch.sin(frequencies*position_of_tokens).unsqueeze(0)

# x is the input embedding of d= 64
x_0 = x[:, :, 0::2]
x_1 = x[:, :, 1::2]

Rope_x0 = x_0*cos_frequencies-x_1*sin_frequencies
Rope_x1 = x_0*sin_frequencies+x_1*cos_frequencies

stacked_pair = torch.stack([Rope_x0, Rope_x1], dim=-1)
final_pos_vec = stacked_pair.flatten(-2)

In [27]:
stacked_pair.size()
final_pos_vec.size()

torch.Size([4, 1024, 64])

Completed writing a tokenizer(used tiktoken and) and divided the input into the size[B, N] and then embedded the token into a 64d and then passed the embedded token through the Rope

In [92]:
#MHA
batch_size = 4
sequence_length = 1024
d_embed = 64
no_of_heads = 4
noh_d = d_embed//no_of_heads

Query = nn.Linear(d_embed, d_embed)
Key = nn.Linear(d_embed, d_embed)
Value = nn.Linear(d_embed, d_embed)
out_proj = nn.Linear(d_embed, d_embed)


q  = Query(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
k = Key(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
v = Value(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)

#####Attention_Mechanism#######
attention_scores = ((q@k.transpose(-1, -2))/(noh_d)**0.5)#shape 4, 4, 1024, 1024]
####Masking(self attention)######
masking_matrix = torch.tril(torch.ones(sequence_length, sequence_length))
masked_attention_scores = attention_scores.masked_fill(masking_matrix ==0, float('-inf'))
########################################--Masked Attention--###############
sliding_window = 256
masking_matrix_1 = torch.tril(torch.ones(sequence_length, sequence_length), diagonal=0)
masking_matrix_2 = torch.tril(torch.ones(sequence_length, sequence_length), diagonal=-sliding_window)
final_mat = (masking_matrix_1-masking_matrix_2)
local_masked_attention_scores = attention_scores.masked_fill(final_mat ==0, float('-inf'))
soft_maxed_local_scores = torch.softmax(local_masked_attention_scores, dim=-1)
#############################-------#############################
soft_maxed_scores = torch.softmax(attention_scores, dim=-1)
final_attention_scores = (soft_maxed_scores@v).transpose(1, 2)#shape [4, 1024, 4, 16]
final_reshaped_score = final_attention_scores.reshape(batch_size, sequence_length, d_embed)
output_projection = out_proj(final_reshaped_score)

In [89]:
#testing local attention
sliding_window = 2
masking_matrix_1 = torch.tril(torch.ones(10, 10), diagonal=0)
masking_matrix_2 = torch.tril(torch.ones(10, 10), diagonal=-sliding_window)
final_mat = (masking_matrix_1-masking_matrix_2)
final_mat

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.]])

In [15]:
class ChatGpt(nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding_table = nn.Embedding(vocab_size, d_embed)


  def forward(self, torched_sequence):
    self.Embedded_table = self.Embedding_table(torched_sequence)
    return self.Embedded_table

  # def Rope_Encodding(self, X_input):

SyntaxError: incomplete input (2513783445.py, line 11)

In [93]:
soft_maxed_local_scores

tensor([[[[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.6105, 0.3895, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.2849, 0.3695, 0.3456,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0025, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0038, 0.0038, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0019, 0.0038, 0.0064]],

         [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.5421, 0.4579, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.3532, 0.1774, 0.4693,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0038, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0029, 0.0033, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0035, 0.0032, 0.0034]],

         [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.4579, 0.5421, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.4348, 0.3562, 0.2090,  ..., 0